# gas-sim-pro · Training Notebook
**Always open fresh from GitHub** — never reuse old Colab tabs.

Run all cells top to bottom. Cell 6b (Optuna) is optional.
Cells 8-11 skip automatically if gate fails.


In [ ]:
# ── Cell 1 — Config + auth + registry ───────────────────────────────────
PROJECT_ID = 'gas-sim-pro-ii'
BUCKET     = f'{PROJECT_ID}-gas-sim-data'

from google.colab import auth
auth.authenticate_user()

from google.cloud import storage
import json

gcs    = storage.Client(project=PROJECT_ID)
bucket = gcs.bucket(BUCKET)

def read_registry():
    return json.loads(bucket.blob('model_registry.json').download_as_text())

reg = read_registry()
print('── Registry ──────────────────────────────')
for k in ['last_trained','last_data_upload','feature_version','mae','previous_mae','rows_trained_on','gate_status']:
    print(f'  {k}: {reg.get(k)}')
print('──────────────────────────────────────────')


In [ ]:
# ── Cell 2 — Install dependencies ───────────────────────────────────────
!pip install -q xgboost wandb pyarrow
print('Done.')


In [ ]:
# ── Cell 3 — Load features from GCS ─────────────────────────────────────
import pandas as pd
import numpy as np

blobs = list(gcs.bucket(BUCKET).list_blobs(prefix='features/latest/'))
paths = [f'gs://{BUCKET}/{b.name}' for b in blobs if b.name.endswith('.parquet')]
print(f'Found {len(paths)} parquet file(s)')
assert len(paths) > 0, 'No Parquet files found — run cloud.sh option 8 first'

df = pd.concat(
    [pd.read_parquet(p, storage_options={'token': 'google_default'}) for p in paths],
    ignore_index=True
)
assert len(df) >= 500, f'Only {len(df)} rows — need at least 500'
print(f'Loaded {len(df):,} rows · {len(df.columns)} columns')
print(f'sensor_delta: min={df["sensor_delta"].min():.4f}  mean={df["sensor_delta"].mean():.4f}  max={df["sensor_delta"].max():.4f}')
if 'n_walls' in df.columns:
    print(f'Wall features present — n_walls mean={df["n_walls"].mean():.1f}')


In [ ]:
# ── Cell 4 — Feature matrix + train/val split ────────────────────────────
# 34 features selected via forward selection + backward elimination
# (correlation analysis + XGBoost importance on 63k rows at threshold=0.15)
from sklearn.model_selection import train_test_split

TARGETS = ['target_centroid_row', 'target_centroid_col']
SIGNAL_THRESHOLD = 0.15

# Apply signal threshold
df_t = df[df['sensor_delta'] >= SIGNAL_THRESHOLD].reset_index(drop=True)

print(f'                      sensor_mean mean:   {df_t["sensor_mean"].mean():.4f}')
print(f'                      t1_t2_ratio mean:   {df_t["t1_t2_ratio"].mean():.2f}')

print(f'After signal filter (>= {SIGNAL_THRESHOLD}): {len(df_t):,} rows')

# ── Parquet columns used directly ────────────────────────────────────────
PARQUET_FEATURES = [
    'top1_col', 'top1_row',
    'top3_centroid_row', 'top3_centroid_col',   # if in parquet
    'wind_x', 'wind_y', 'wind_angle',
    'coverage_ratio', 'centroid_col',
    'top2_reading', 'top3_col', 'sensor_mean',
    'top3_reading', 'top2_col',
    'open_path_ratio', 'walls_blocking_top1',
    'top3_row', 'top1_reading',
    't1_t2_vec_row', 't1_t2_vec_col',           # if in parquet
    'distance_to_boundary', 'wall_density',
    't1_t2_dist',                               # if in parquet
    'wall_asymmetry_col', 'sensor_count',
    'walls_q1', 'wall_spread_row',
    'reading_variance', 'wall_asymmetry_row',
]
avail_parquet = [f for f in PARQUET_FEATURES if f in df_t.columns]
df_f = df_t[avail_parquet].copy()

# ── Always-computed derived features ─────────────────────────────────────
df_f['wind_corr_row'] = df_t['centroid_row'] - df_t['wind_y'] * 5
df_f['wind_corr_col'] = df_t['centroid_col'] - df_t['wind_x'] * 5
df_f['disp_row'] = df_t['top1_row'] - df_t['centroid_row']
df_f['disp_col'] = df_t['top1_col'] - df_t['centroid_col']

# Triangulation — compute if not already in parquet
if 'top3_centroid_row' not in df_t.columns:
    total = df_t['top1_reading'] + df_t['top2_reading'] + df_t['top3_reading'] + 1e-9
    df_f['top3_centroid_row'] = (df_t['top1_row']*df_t['top1_reading'] + df_t['top2_row']*df_t['top2_reading'] + df_t['top3_row']*df_t['top3_reading']) / total
    df_f['top3_centroid_col'] = (df_t['top1_col']*df_t['top1_reading'] + df_t['top2_col']*df_t['top2_reading'] + df_t['top3_col']*df_t['top3_reading']) / total
if 't1_t2_ratio' not in df_t.columns:
    df_f['t1_t2_ratio'] = (df_t['top1_reading'] / (df_t['top2_reading'] + 1e-9)).clip(0, 100)
if 't1_t2_vec_row' not in df_t.columns:
    df_f['t1_t2_vec_row'] = df_t['top1_row'] - df_t['top2_row']
    df_f['t1_t2_vec_col'] = df_t['top1_col'] - df_t['top2_col']
if 't1_t2_ratio' not in df_t.columns:
    df_f['t1_t2_ratio'] = (df_t['top1_reading'] / (df_t['top2_reading'] + 1e-9)).clip(0, 10000)
    df_f['t1_t3_ratio'] = (df_t['top1_reading'] / (df_t['top3_reading'] + 1e-9)).clip(0, 10000)
if 't1_t2_dist' not in df_t.columns:
    df_f['t1_t2_dist'] = np.sqrt(
        (df_t['top1_row']-df_t['top2_row'])**2 +
        (df_t['top1_col']-df_t['top2_col'])**2).clip(0, 142)

# Wall-derived
if 'n_walls' in df_t.columns:
    if 'wall_asymmetry_col' not in df_t.columns:
        df_f['wall_asymmetry_col'] = df_t['walls_q1'] + df_t['walls_q3'] - df_t['walls_q2'] - df_t['walls_q4']
        df_f['wall_asymmetry_row'] = df_t['walls_q1'] + df_t['walls_q2'] - df_t['walls_q3'] - df_t['walls_q4']

# ── Sanitise ──────────────────────────────────────────────────────────────
for col in ['t1_t2_ratio', 't1_t3_ratio']:
    if col in df_f.columns:
        df_f[col] = df_f[col].replace([np.inf, -np.inf], np.nan).fillna(0).clip(0, 100)
df_f['t1_t2_dist'] = df_f['t1_t2_dist'].clip(0, 142) if 't1_t2_dist' in df_f.columns else df_f.get('t1_t2_dist', 0)

X = df_f.replace([np.inf, -np.inf], 0).fillna(0).values.astype(np.float32)
y = df_t[TARGETS].values.astype(np.float32)

assert not np.isinf(X).any(), 'inf in X'
assert not np.isnan(X).any(), 'nan in X'

X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.15, random_state=42)
FEATURES = list(df_f.columns)
print(f'Features: {len(FEATURES)}')
print(f'Train: {len(X_tr):,}  Val: {len(X_val):,}')
print(f'Feature list: {FEATURES}')


In [ ]:
# ── Cell 5 — W&B init (optional, never blocks) ───────────────────────────
import wandb
WANDB_ACTIVE = False
run = None
try:
    run = wandb.init(
        project='gas-sim-pro', anonymous='allow', mode='offline',
        config={'features': FEATURES, 'n_rows': len(df_t),
                'threshold': SIGNAL_THRESHOLD,
                'feature_version': reg['feature_version']}
    )
    WANDB_ACTIVE = True
    print('W&B initialised (offline)')
except Exception as e:
    print(f'W&B unavailable: {e}')


In [ ]:
# ── Cell 6 — Train XGBoost ───────────────────────────────────────────────
import xgboost as xgb
from sklearn.multioutput import MultiOutputRegressor

BASE_PARAMS = dict(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, n_jobs=-1, verbosity=0
)

model_centroid = MultiOutputRegressor(xgb.XGBRegressor(**BASE_PARAMS))
model_centroid.fit(X_tr, y_tr)

y_pred = model_centroid.predict(X_val)
mae    = float(np.mean(np.abs(y_pred - y_val)))
model  = model_centroid
mae_centroid = mae
mae_nearest  = mae
mae_count    = 0.0

if WANDB_ACTIVE and run:
    wandb.log({'val_mae': mae, 'n_features': len(FEATURES), 'threshold': SIGNAL_THRESHOLD})

print(f'Val MAE: {mae:.4f} cells')
print(f'Primary metric (centroid MAE): {mae:.4f}')


In [ ]:
# ── Cell 6b — Optuna hyperparameter search (optional) ────────────────────
# Set RUN_OPTUNA=True to search. ~20-40 min on Colab free tier.
RUN_OPTUNA = False
N_TRIALS   = 40

if RUN_OPTUNA:
    !pip install -q optuna
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    def objective(trial):
        params = dict(
            n_estimators     = trial.suggest_int('n_estimators', 100, 1000),
            max_depth        = trial.suggest_int('max_depth', 3, 10),
            learning_rate    = trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            subsample        = trial.suggest_float('subsample', 0.5, 1.0),
            colsample_bytree = trial.suggest_float('colsample_bytree', 0.5, 1.0),
            min_child_weight = trial.suggest_int('min_child_weight', 1, 10),
            gamma            = trial.suggest_float('gamma', 0, 5),
            random_state=42, n_jobs=-1, verbosity=0
        )
        m = MultiOutputRegressor(xgb.XGBRegressor(**params))
        m.fit(X_tr, y_tr)
        return float(np.mean(np.abs(m.predict(X_val) - y_val)))

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
    best = study.best_params
    print(f'Best MAE: {study.best_value:.4f}  Params: {best}')
    bucket.blob('registry/hparam_best.json').upload_from_string(
        json.dumps({'best_params': best, 'best_mae': study.best_value,
                    'n_trials': N_TRIALS, 'n_rows': len(df_t),
                    'searched_at': __import__('datetime').datetime.utcnow().isoformat()},
                   indent=2), content_type='application/json')
    model_centroid = MultiOutputRegressor(xgb.XGBRegressor(**best, random_state=42, n_jobs=-1))
    model_centroid.fit(X_tr, y_tr)
    y_pred = model_centroid.predict(X_val)
    mae = float(np.mean(np.abs(y_pred - y_val)))
    model = model_centroid
    mae_centroid = mae
    print(f'Final MAE with best params: {mae:.4f}')
else:
    try:
        saved = json.loads(bucket.blob('registry/hparam_best.json').download_as_text())
        if saved.get('n_rows', 0) >= len(df_t) * 0.5:
            best = saved['best_params']
            print(f'Using saved hyperparameters (searched on {saved["n_rows"]:,} rows)')
            model_centroid = MultiOutputRegressor(xgb.XGBRegressor(**best, random_state=42, n_jobs=-1))
            model_centroid.fit(X_tr, y_tr)
            y_pred = model_centroid.predict(X_val)
            mae = float(np.mean(np.abs(y_pred - y_val)))
            model = model_centroid
            mae_centroid = mae
            print(f'MAE with saved params: {mae:.4f}')
        else:
            print('Saved params from too few rows — using Cell 6 defaults.')
    except Exception:
        print('No saved hyperparameters — using Cell 6 defaults.')


In [ ]:
# ── Cell 7 — MAE gate (advisory, never stops execution) ──────────────────
reg = read_registry()
prod_mae = reg.get('mae') or float('inf')

print('── MAE Gate Assessment ──────────────────────────────────────')
print(f'  New MAE    : {mae:.4f} cells')
print(f'  Prod MAE   : {prod_mae:.4f} cells')
print(f'  Threshold  : {prod_mae * 0.98:.4f}')
print()

if prod_mae == float('inf'):
    GATE_STATUS = 'first_model'
    print('✅ First model — will deploy.')
elif mae < prod_mae * 0.98:
    GATE_STATUS = 'passed'
    print(f'✅ GATE PASSED: {(prod_mae-mae)/prod_mae*100:.1f}% improvement.')
elif mae < prod_mae:
    GATE_STATUS = 'marginal'
    print(f'⚠️  MARGINAL: {(prod_mae-mae)/prod_mae*100:.1f}% better but below 2% threshold.')
else:
    GATE_STATUS = 'failed'
    print(f'❌ FAILED: {(mae-prod_mae)/prod_mae*100:.1f}% worse than production.')
    print('   Possible causes: different threshold/features, same data, distribution shift.')
    print('   If prod MAE was from leaked model: cloud.sh option 5 to reset.')
    print('   Cells 8-9 skipped. Prod model unchanged.')

if WANDB_ACTIVE and run:
    wandb.log({'gate_status': GATE_STATUS, 'prod_mae': prod_mae})
print('─────────────────────────────────────────────────────────────')


In [ ]:
# ── Cell 8 — Export model ────────────────────────────────────────────────
import os, datetime, joblib

if GATE_STATUS not in ('passed', 'first_model'):
    print(f'⏭  Skipped — gate: {GATE_STATUS}')
    VERSION = None
else:
    RUN_ID  = run.id if WANDB_ACTIVE and run else datetime.datetime.now().strftime('%H%M%S')
    VERSION = f"v{datetime.date.today().strftime('%Y%m%d')}-{RUN_ID}"
    os.makedirs(f'/tmp/{VERSION}', exist_ok=True)
    joblib.dump(model_centroid, f'/tmp/{VERSION}/model.joblib')
    print(f'✓ Exported: {VERSION}  MAE: {mae:.4f}  Features: {len(FEATURES)}')


In [ ]:
# ── Cell 9 — Push to GCS ─────────────────────────────────────────────────
if GATE_STATUS not in ('passed', 'first_model') or VERSION is None:
    print(f'⏭  Skipped — gate: {GATE_STATUS}')
else:
    blob = bucket.blob(f'models/{VERSION}/model.joblib')
    blob.upload_from_filename(f'/tmp/{VERSION}/model.joblib')
    print(f'✓ Uploaded: models/{VERSION}/model.joblib')
    if WANDB_ACTIVE and run:
        try:
            wandb.log_artifact(f'/tmp/{VERSION}/model.joblib', name='model', type='model')
        except Exception as e:
            print(f'  W&B artifact skipped: {e}')


In [ ]:
# ── Cell 10 — Update registry ────────────────────────────────────────────
import datetime as dt

prev_version = reg.get('latest_version')
prev_mae_val = reg.get('mae')

if GATE_STATUS in ('passed', 'first_model') and VERSION is not None:
    reg.update({
        'latest_version':   VERSION,
        'previous_version': prev_version,
        'last_trained':     dt.datetime.now(dt.timezone.utc).isoformat(),
        'model_path':       f'models/{VERSION}/model.joblib',
        'joblib_path':      f'models/{VERSION}/model.joblib',
        'mae':              mae,
        'previous_mae':     prev_mae_val,
        'rows_trained_on':  len(df_t),
        'feature_version':  reg['feature_version'],
        'gate_status':      GATE_STATUS,
        'n_features':       len(FEATURES),
        'signal_threshold': SIGNAL_THRESHOLD,
    })
else:
    reg.update({
        'last_trained':    dt.datetime.now(dt.timezone.utc).isoformat(),
        'gate_status':     GATE_STATUS,
        'rows_trained_on': len(df_t),
    })

reg_blob = bucket.blob('model_registry.json')
reg_blob.upload_from_string(json.dumps(reg, indent=2), content_type='application/json')
reg_blob.cache_control = 'no-cache, no-store, max-age=0'
reg_blob.patch()

result = {
    'status':     GATE_STATUS,
    'mae':        mae,
    'prod_mae':   prod_mae,
    'version':    VERSION,
    'trained_at': dt.datetime.now(dt.timezone.utc).isoformat(),
    'rows':       len(df_t),
    'n_features': len(FEATURES),
}
r_blob = bucket.blob('registry/last_training_result.json')
r_blob.upload_from_string(json.dumps(result, indent=2, default=str), content_type='application/json')
r_blob.cache_control = 'no-cache, no-store, max-age=0'
r_blob.patch()

if WANDB_ACTIVE and run:
    wandb.finish()

print()
print('── Training Summary ─────────────────────────────────────────')
print(f'  Gate:      {GATE_STATUS}')
print(f'  MAE:       {mae:.4f}')
print(f'  Version:   {VERSION}')
print(f'  Rows:      {len(df_t):,}')
print(f'  Features:  {len(FEATURES)}')
print()
if GATE_STATUS in ('passed', 'first_model'):
    print('🚀 Deploy function will trigger automatically (~5 min).')
    print('   TRAIN button will flash green when deployed.')
elif GATE_STATUS == 'marginal':
    print('⚠️  Below threshold. Generate more data and retrain.')
    print('   TRAIN button will flash yellow.')
else:
    print('❌ Prod model unchanged.')
    print('   1. cloud.sh option 8 (refresh features)')
    print('   2. Generate more data')
    print('   3. cloud.sh option 5 if prod MAE was from leaked model')
    print('   TRAIN button will flash yellow.')
print('─────────────────────────────────────────────────────────────')


In [ ]:
# ── Cell 11 — Verify predictions ─────────────────────────────────────────
if GATE_STATUS not in ('passed', 'first_model') or VERSION is None:
    print(f'⏭  Skipped — gate: {GATE_STATUS}')
else:
    pred = model_centroid.predict(X_val[:5])
    print('Predictions (first 5 rows):')
    print(f'  {"pred_row":>8}  {"pred_col":>8}  {"true_row":>8}  {"true_col":>8}  {"err":>6}')
    for i in range(5):
        err = abs(pred[i][0]-y_val[i][0]) + abs(pred[i][1]-y_val[i][1])
        print(f'  {pred[i][0]:8.1f}  {pred[i][1]:8.1f}  {y_val[i][0]:8.1f}  {y_val[i][1]:8.1f}  {err:6.2f}')
    print('Verification complete.')
